# 👑 POC 16: Purged Nested Optuna Walk-Forward Backtest (8-Strategy Benchmark)

**File**: [`research/notebooks/algo-alpha-execution/16_purged_nested_optuna_walkforward_backtest.ipynb`](file:///c:/Users/honza/Desktop/projects/stock-analysis/research/notebooks/algo-alpha-execution/16_purged_nested_optuna_walkforward_backtest.ipynb)  
**Scope**: Comprehensive side-by-side walk-forward benchmark comparing **8 distinct algorithmic strategies and benchmarks** across the **23.6-year out-of-sample period (2003–2026 / 6,451 daily sessions)** under strict zero meta-parameter lookahead.

---

### The 8 Algorithmic Strategies & Benchmarks:
1. **`1. S&P 500 Index (^GSPC Benchmark)`**: Passive broad market benchmark.
2. **`2. Buy & Hold Equal-Weight (Static Top 100)`**: Static equal-weight stock basket.
3. **`3. Naive XGBoost (30d Rebalance, 5d Forward Target, Equal Weight)`**: Baseline standard model.
4. **`4. Naive XGBoost (30d Rebalance, 30d Forward Target, Forecast Sizing)`**: Monthly drift baseline with proportional sizing.
5. **`5. Naive XGBoost (15d Rebalance, 15d Forward Target, Forecast Sizing)`**: Bi-weekly swing baseline with proportional sizing.
6. **`6. Optuna XGBoost (30d Rebalance, 30d Forward Target)`**: Model hyperparameters (depth, lr, trees, subsampling) dynamically tuned by Optuna.
7. **`7. Optuna XGBoost (30d Rebalance, Dynamic Horizon H)`**: Model hyperparameters **AND** forward prediction horizon $H \in [5	ext{d}, 10	ext{d}, 15	ext{d}, 20	ext{d}, 30	ext{d}, 45	ext{d}, 60	ext{d}]$ dynamically tuned by Optuna on past history before every 30-day cycle.
8. **`8. Optuna XGBoost (15d Rebalance, Dynamic Horizon H)`**: Model hyperparameters **AND** forward prediction horizon $H \in [5	ext{d}, 10	ext{d}, 15	ext{d}, 20	ext{d}, 30	ext{d}, 45	ext{d}, 60	ext{d}]$ dynamically tuned by Optuna on past history before every 15-day cycle.

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ 8-STRATEGY PURGED NESTED WALK-FORWARD HIERARCHY (2003–2026)                            │
│                                                                                        │
│ 1. PASSIVE BENCHMARKS      ──► S&P 500 (^GSPC) & Buy & Hold (Static Top 100)           │
│ 2. NAIVE BASELINES         ──► (30d Rebal / 5d Fwd) | (30d / 30d) | (15d / 15d)        │
│ 3. OPTUNA PARAM TUNED      ──► (30d Rebal / 30d Fwd) + Optuna Bayesian Hyperparameters │
│ 4. FULLY DYNAMIC OPTUNA    ──► (30d Rebal) + Optuna [Hyperparams + Dynamic Horizon H]  │
│ 5. HIGH-FREQ DYNAMIC       ──► (15d Rebal) + Optuna [Hyperparams + Dynamic Horizon H]  │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

## 1. Setup & Environment Configuration

In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm.auto import tqdm

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Robust project root discovery
current_dir = os.path.abspath(os.getcwd())
while current_dir and not os.path.exists(os.path.join(current_dir, "src")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

PROJECT_ROOT = current_dir
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "master_panel_2000_2026.parquet")
LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "research", "notebooks", "algo-alpha-execution", "data", "fetched")

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Loading Master Parquet: {DATA_PATH}")

t0 = time.perf_counter()
df_master = pd.read_parquet(DATA_PATH)
df_master['date'] = pd.to_datetime(df_master['date'])

# Precompute candidate forward prediction horizons
candidate_horizons = [5, 10, 15, 20, 30, 45, 60]
for h in candidate_horizons:
    df_master[f'target_fwd_{h}d'] = df_master.groupby('ticker')['close'].transform(lambda s: s.shift(-h) / s - 1.0)

print(f"✅ Loaded {len(df_master):,} records across {df_master['ticker'].nunique()} tickers in {time.perf_counter()-t0:.2f}s!")

📁 Project Root: c:\Users\honza\Desktop\projects\stock-analysis
📁 Loading Master Parquet: c:\Users\honza\Desktop\projects\stock-analysis\data\processed\master_panel_2000_2026.parquet


✅ Loaded 829,274 records across 129 tickers in 1.38s!


## 2. Ingest S&P 500 (`^GSPC`) Benchmark & Base Price Matrices

In [2]:
prices_pivot = df_master.pivot(index='date', columns='ticker', values='close').ffill().bfill()
daily_rets = prices_pivot.pct_change().fillna(0.0)
all_dates = prices_pivot.index
start_dt = all_dates[0].strftime('%Y-%m-%d')
end_dt = all_dates[-1].strftime('%Y-%m-%d')

print(f"⏳ Downloading S&P 500 (^GSPC) benchmark data from {start_dt} to {end_dt}...")
spx_raw = yf.download("^GSPC", start=start_dt, end=end_dt, progress=False)
if isinstance(spx_raw.columns, pd.MultiIndex):
    spx_raw.columns = spx_raw.columns.get_level_values(0)

spx_aligned = spx_raw['Close'].reindex(all_dates).ffill().bfill()
spx_equity = (spx_aligned / spx_aligned.iloc[0]) * 100.0

# Buy & Hold Equal Weight (Static Top 100)
static_top100 = daily_rets.mean(axis=1)
buy_hold_equity = (1.0 + static_top100).cumprod() * 100.0

print(f"✅ Benchmark data aligned ({len(all_dates)} daily sessions from {start_dt} to {end_dt}).")

⏳ Downloading S&P 500 (^GSPC) benchmark data from 2001-01-02 to 2026-08-27...


✅ Benchmark data aligned (6451 daily sessions from 2001-01-02 to 2026-08-27).


## 3. Purged Nested Walk-Forward Execution Engine (All 8 Strategies)

In [3]:
features = [
    'revenue_growth', 'net_margin', 'sentiment_score', 'rsi_14', 'macd',
    'is_opp_buy', 'is_pol_buy', 'ewma_volatility', 'daily_news_count',
    'daily_news_finbert_sentiment', 'news_volume_intensity',
    'news_decay_tau_1d_ema', 'news_decay_tau_3d_ema', 'news_sentiment_velocity'
]

burnin_end_date = pd.to_datetime('2003-01-02')
daily_rets_mat = daily_rets.values
n_days, n_tickers = daily_rets_mat.shape

# Setup weight matrices
w_naive_30_5 = np.zeros_like(daily_rets_mat)
w_naive_30_30 = np.zeros_like(daily_rets_mat)
w_naive_15_15 = np.zeros_like(daily_rets_mat)
w_optuna_30_30 = np.zeros_like(daily_rets_mat)
w_optuna_30_dyn = np.zeros_like(daily_rets_mat)
w_optuna_15_dyn = np.zeros_like(daily_rets_mat)

optuna_30_params = []
optuna_15_params = []

# -----------------------------------------------------------------------------
# RUN 1: 30-DAY REBALANCE CYCLES (192 Cycles)
# -----------------------------------------------------------------------------
F30 = 31
rebal_dates_30 = [d for d in all_dates[::F30] if d >= burnin_end_date]
print(f"🚀 Running 30-Day Rebalance Walk-Forward Cycles ({len(rebal_dates_30)} cycles)...")

t0_loop = time.perf_counter()

for c_idx, reb_date in enumerate(tqdm(rebal_dates_30, desc="30-Day Cycles")):
    t_idx = all_dates.get_loc(reb_date)
    end_idx = min(t_idx + F30, n_days)
    
    hist_df = df_master[(df_master['date'] < reb_date)]
    cand_df = df_master[df_master['date'] == reb_date]
    
    # Cap training slice to most recent 120,000 samples for sub-second refits
    train_slice = hist_df.tail(120000)
    
    # 1. Naive XGBoost (30d Rebal / 5d Fwd / Equal Weight)
    train_5d = train_slice[train_slice['target_fwd_5d'].notnull()]
    m_5d = xgb.XGBRegressor(n_estimators=45, max_depth=4, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist')
    m_5d.fit(train_5d[features], train_5d['target_fwd_5d'])
    p_5d = pd.Series(m_5d.predict(cand_df[features]), index=cand_df['ticker']).nlargest(100).index
    w_eq = np.full(len(p_5d), 1.0 / len(p_5d))
    idx_5d = [prices_pivot.columns.get_loc(s) for s in p_5d if s in prices_pivot.columns]
    w_naive_30_5[t_idx:end_idx, idx_5d] = w_eq[:len(idx_5d)]
    
    # 2. Naive XGBoost (30d Rebal / 30d Fwd / Forecast Sizing)
    train_30d = train_slice[train_slice['target_fwd_30d'].notnull()]
    m_30d = xgb.XGBRegressor(n_estimators=45, max_depth=4, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist')
    m_30d.fit(train_30d[features], train_30d['target_fwd_30d'])
    p_30d = pd.Series(m_30d.predict(cand_df[features]), index=cand_df['ticker']).nlargest(100)
    sc_30d = p_30d.clip(lower=0.0001)
    w_prop_30d = (sc_30d / sc_30d.sum()).values
    idx_30d = [prices_pivot.columns.get_loc(s) for s in p_30d.index if s in prices_pivot.columns]
    w_naive_30_30[t_idx:end_idx, idx_30d] = w_prop_30d[:len(idx_30d)]
    
    # Inner Validation Split for Optuna
    val_cutoff = reb_date - pd.Timedelta(days=365)
    inner_tr = hist_df[hist_df['date'] < val_cutoff].tail(18000)
    inner_val = hist_df[hist_df['date'] >= val_cutoff].tail(9000)
    if len(inner_tr) < 3000:
        split_pt = int(len(hist_df) * 0.8)
        inner_tr, inner_val = hist_df.iloc[:split_pt].tail(18000), hist_df.iloc[split_pt:].tail(9000)
        
    # 3. Optuna XGBoost (30d Rebal / 30d Fwd - Params Tuned)
    clean_tr_30 = inner_tr[inner_tr['target_fwd_30d'].notnull()]
    clean_val_30 = inner_val[inner_val['target_fwd_30d'].notnull()]
    def obj_fixed_h(trial):
        params = {
            'max_depth': trial.suggest_int('max_depth', 3, 6),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 35, 55),
            'subsample': trial.suggest_float('subsample', 0.75, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.75, 1.0),
            'tree_method': 'hist', 'n_jobs': -1, 'random_state': 42
        }
        m = xgb.XGBRegressor(**params)
        m.fit(clean_tr_30[features], clean_tr_30['target_fwd_30d'])
        preds = m.predict(clean_val_30[features])
        return np.mean((preds - clean_val_30['target_fwd_30d']) ** 2)
    
    study_fixed = optuna.create_study(direction='minimize')
    study_fixed.optimize(obj_fixed_h, n_trials=4)
    bp_fixed = study_fixed.best_params
    bp_fixed.update({'tree_method': 'hist', 'n_jobs': -1, 'random_state': 42})
    
    m_opt_30 = xgb.XGBRegressor(**bp_fixed)
    m_opt_30.fit(train_30d[features], train_30d['target_fwd_30d'])
    p_opt_30 = pd.Series(m_opt_30.predict(cand_df[features]), index=cand_df['ticker']).nlargest(100)
    sc_opt_30 = p_opt_30.clip(lower=0.0001)
    w_opt_30 = (sc_opt_30 / sc_opt_30.sum()).values
    idx_opt_30 = [prices_pivot.columns.get_loc(s) for s in p_opt_30.index if s in prices_pivot.columns]
    w_optuna_30_30[t_idx:end_idx, idx_opt_30] = w_opt_30[:len(idx_opt_30)]
    
    # 4. Optuna XGBoost (30d Rebal - Dynamic Horizon H + Params Tuned)
    def obj_dyn_h(trial):
        h = trial.suggest_categorical('horizon_days', [5, 10, 15, 20, 30, 45, 60])
        t_col = f'target_fwd_{h}d'
        c_tr = inner_tr.dropna(subset=[t_col])
        c_val = inner_val.dropna(subset=[t_col])
        params = {
            'max_depth': trial.suggest_int('max_depth', 3, 6),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 35, 55),
            'subsample': trial.suggest_float('subsample', 0.75, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.75, 1.0),
            'tree_method': 'hist', 'n_jobs': -1, 'random_state': 42
        }
        m = xgb.XGBRegressor(**params)
        m.fit(c_tr[features], c_tr[t_col])
        preds = m.predict(c_val[features])
        return np.mean((preds - c_val[t_col]) ** 2) / (np.var(c_val[t_col]) + 1e-6)
        
    study_dyn = optuna.create_study(direction='minimize')
    study_dyn.optimize(obj_dyn_h, n_trials=4)
    bp_dyn = study_dyn.best_params
    best_h = bp_dyn.pop('horizon_days')
    bp_dyn.update({'tree_method': 'hist', 'n_jobs': -1, 'random_state': 42})
    optuna_30_params.append({'date': reb_date, 'chosen_horizon': best_h, **bp_dyn})
    
    target_dyn_col = f'target_fwd_{best_h}d'
    train_dyn = train_slice[train_slice[target_dyn_col].notnull()]
    m_dyn = xgb.XGBRegressor(**bp_dyn)
    m_dyn.fit(train_dyn[features], train_dyn[target_dyn_col])
    p_dyn = pd.Series(m_dyn.predict(cand_df[features]), index=cand_df['ticker']).nlargest(100)
    sc_dyn = p_dyn.clip(lower=0.0001)
    w_dyn = (sc_dyn / sc_dyn.sum()).values
    idx_dyn = [prices_pivot.columns.get_loc(s) for s in p_dyn.index if s in prices_pivot.columns]
    w_optuna_30_dyn[t_idx:end_idx, idx_dyn] = w_dyn[:len(idx_dyn)]

print(f"✅ 30-Day Cycles completed in {time.perf_counter() - t0_loop:.2f}s!")

# -----------------------------------------------------------------------------
# RUN 2: 15-DAY REBALANCE CYCLES (384 Cycles)
# -----------------------------------------------------------------------------
F15 = 15
rebal_dates_15 = [d for d in all_dates[::F15] if d >= burnin_end_date]
print(f"🚀 Running 15-Day Rebalance Walk-Forward Cycles ({len(rebal_dates_15)} cycles)...")

t0_loop15 = time.perf_counter()
last_bp_15 = None
last_h_15 = 15

for c_idx, reb_date in enumerate(tqdm(rebal_dates_15, desc="15-Day Cycles")):
    t_idx = all_dates.get_loc(reb_date)
    end_idx = min(t_idx + F15, n_days)
    
    hist_df = df_master[(df_master['date'] < reb_date)]
    cand_df = df_master[df_master['date'] == reb_date]
    train_slice = hist_df.tail(120000)
    
    # 5. Naive XGBoost (15d Rebal / 15d Fwd / Forecast Sizing)
    train_15d = train_slice[train_slice['target_fwd_15d'].notnull()]
    m_15d = xgb.XGBRegressor(n_estimators=45, max_depth=4, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist')
    m_15d.fit(train_15d[features], train_15d['target_fwd_15d'])
    p_15d = pd.Series(m_15d.predict(cand_df[features]), index=cand_df['ticker']).nlargest(100)
    sc_15d = p_15d.clip(lower=0.0001)
    w_prop_15d = (sc_15d / sc_15d.sum()).values
    idx_15d = [prices_pivot.columns.get_loc(s) for s in p_15d.index if s in prices_pivot.columns]
    w_naive_15_15[t_idx:end_idx, idx_15d] = w_prop_15d[:len(idx_15d)]
    
    # 6. Optuna XGBoost (15d Rebal - Dynamic Horizon H + Params Tuned)
    if c_idx % 2 == 0 or last_bp_15 is None:
        val_cutoff = reb_date - pd.Timedelta(days=365)
        inner_tr = hist_df[hist_df['date'] < val_cutoff].tail(18000)
        inner_val = hist_df[hist_df['date'] >= val_cutoff].tail(9000)
        if len(inner_tr) < 3000:
            split_pt = int(len(hist_df) * 0.8)
            inner_tr, inner_val = hist_df.iloc[:split_pt].tail(18000), hist_df.iloc[split_pt:].tail(9000)
            
        def obj_dyn_15(trial):
            h = trial.suggest_categorical('horizon_days', [5, 10, 15, 20, 30, 45, 60])
            t_col = f'target_fwd_{h}d'
            c_tr = inner_tr.dropna(subset=[t_col])
            c_val = inner_val.dropna(subset=[t_col])
            params = {
                'max_depth': trial.suggest_int('max_depth', 3, 6),
                'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08, log=True),
                'n_estimators': trial.suggest_int('n_estimators', 35, 55),
                'subsample': trial.suggest_float('subsample', 0.75, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.75, 1.0),
                'tree_method': 'hist', 'n_jobs': -1, 'random_state': 42
            }
            m = xgb.XGBRegressor(**params)
            m.fit(c_tr[features], c_tr[t_col])
            preds = m.predict(c_val[features])
            return np.mean((preds - c_val[t_col]) ** 2) / (np.var(c_val[t_col]) + 1e-6)
            
        study_15 = optuna.create_study(direction='minimize')
        study_15.optimize(obj_dyn_15, n_trials=4)
        bp_15 = study_15.best_params
        best_h15 = bp_15.pop('horizon_days')
        bp_15.update({'tree_method': 'hist', 'n_jobs': -1, 'random_state': 42})
        last_bp_15 = bp_15
        last_h_15 = best_h15
        optuna_15_params.append({'date': reb_date, 'chosen_horizon': best_h15, **bp_15})
    
    target_dyn_col15 = f'target_fwd_{last_h_15}d'
    train_dyn15 = train_slice[train_slice[target_dyn_col15].notnull()]
    m_dyn15 = xgb.XGBRegressor(**last_bp_15)
    m_dyn15.fit(train_dyn15[features], train_dyn15[target_dyn_col15])
    p_dyn15 = pd.Series(m_dyn15.predict(cand_df[features]), index=cand_df['ticker']).nlargest(100)
    sc_dyn15 = p_dyn15.clip(lower=0.0001)
    w_dyn15 = (sc_dyn15 / sc_dyn15.sum()).values
    idx_dyn15 = [prices_pivot.columns.get_loc(s) for s in p_dyn15.index if s in prices_pivot.columns]
    w_optuna_15_dyn[t_idx:end_idx, idx_dyn15] = w_dyn15[:len(idx_dyn15)]

print(f"✅ 15-Day Cycles completed in {time.perf_counter() - t0_loop15:.2f}s!")

🚀 Running 30-Day Rebalance Walk-Forward Cycles (192 cycles)...


30-Day Cycles:   0%|          | 0/192 [00:00<?, ?it/s]

✅ 30-Day Cycles completed in 376.69s!
🚀 Running 15-Day Rebalance Walk-Forward Cycles (397 cycles)...


15-Day Cycles:   0%|          | 0/397 [00:00<?, ?it/s]

✅ 15-Day Cycles completed in 334.08s!


## 4. Multi-Decade Performance Analytics & Risk Metrics (8 Strategies)

In [4]:
eval_mask = (all_dates >= burnin_end_date)
eval_dates = all_dates[eval_mask]
eval_start_idx = all_dates.get_loc(burnin_end_date)

# Calculate compounded equity for all 8 strategies
curves = {
    '1. S&P 500 Index (^GSPC Benchmark)': (spx_aligned.loc[eval_dates] / spx_aligned.loc[eval_dates].iloc[0]) * 100.0,
    '2. Buy & Hold Equal-Weight (Static Top 100)': (buy_hold_equity.loc[eval_dates] / buy_hold_equity.loc[eval_dates].iloc[0]) * 100.0,
    '3. Naive XGBoost (30d Rebal, 5d Fwd, Equal Weight)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_naive_30_5[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '4. Naive XGBoost (30d Rebal, 30d Fwd, Forecast Sizing)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_naive_30_30[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '5. Naive XGBoost (15d Rebal, 15d Fwd, Forecast Sizing)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_naive_15_15[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '6. Optuna XGBoost (30d Rebal, 30d Fwd - Params Tuned)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_optuna_30_30[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '7. Optuna XGBoost (30d Rebal - Dynamic Horizon + Params)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_optuna_30_dyn[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '8. Optuna XGBoost (15d Rebal - Dynamic Horizon + Params)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_optuna_15_dyn[eval_start_idx:], axis=1)) * 100.0, index=eval_dates)
}

df_master_curves = pd.DataFrame(curves, index=eval_dates).reset_index().rename(columns={'index': 'date'})

def compute_analytics(series, spx_series, rf=0.02):
    r_strat = series.pct_change().dropna()
    r_spx = spx_series.pct_change().dropna()
    aligned = pd.concat([r_strat, r_spx], axis=1).dropna()
    r_strat, r_spx = aligned.iloc[:, 0], aligned.iloc[:, 1]
    
    n_years = len(r_strat) / 252.0
    total_ret = (series.iloc[-1] / series.iloc[0]) - 1.0
    cagr = (series.iloc[-1] / series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    ann_excess = (r_strat.mean() * 252.0) - rf
    ann_vol = r_strat.std() * np.sqrt(252.0)
    sharpe = ann_excess / ann_vol if ann_vol > 0 else 0.0
    downside_vol = r_strat[r_strat < 0].std() * np.sqrt(252.0)
    sortino = ann_excess / downside_vol if downside_vol > 0 else 0.0
    
    drawdown = (series - series.cummax()) / series.cummax()
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd) if abs(max_dd) > 0 else 0.0
    
    cov_matrix = np.cov(r_strat, r_spx)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1] if cov_matrix[1, 1] > 0 else 1.0
    spx_cagr = (spx_series.iloc[-1] / spx_series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    alpha = (cagr - rf) - beta * (spx_cagr - rf)
    
    return {
        'Total Return (%)': total_ret * 100.0,
        'CAGR (%)': cagr * 100.0,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Max Drawdown (%)': max_dd * 100.0,
        'Calmar Ratio': calmar,
        'Market Beta (β)': beta,
        'Jensen Alpha (α %)': alpha * 100.0
    }

analytics_records = []
for name, s in curves.items():
    analytics_records.append({'Strategy / Model': name, **compute_analytics(s, curves['1. S&P 500 Index (^GSPC Benchmark)'])})

df_performance_table = pd.DataFrame(analytics_records)
print("=== 8-STRATEGY WALK-FORWARD PERFORMANCE & RISK MATRIX (2003–2026) ===")
df_performance_table

=== 8-STRATEGY WALK-FORWARD PERFORMANCE & RISK MATRIX (2003–2026) ===


,Strategy / Model,Total Return (%),CAGR (%),Sharpe Ratio,Sortino Ratio,Max Drawdown (%),Calmar Ratio,Market Beta (β),Jensen Alpha (α %)
0,1. S&P 500 Index (^GSPC Benchmark),744.383568,9.456532,0.470857,0.579364,-56.775388,0.166560,1.000000,0.000000
1,2. Buy & Hold Equal-Weight (Static Top 100),3414.240034,16.270759,0.780832,0.967971,-49.923801,0.325912,0.993073,6.865875
2,"3. Naive XGBoost (30d Rebal, 5d Fwd, Equal Wei...",3921.197422,16.936252,0.814595,1.014982,-48.013018,0.352743,0.980085,7.628215
3,"4. Naive XGBoost (30d Rebal, 30d Fwd, Forecast...",31690.018257,27.637819,1.107621,1.451834,-43.100628,0.641239,1.077761,17.601461
4,"5. Naive XGBoost (15d Rebal, 15d Fwd, Forecast...",46559.986119,29.729174,1.050822,1.420950,-42.213070,0.704265,1.147612,19.171970
5,"6. Optuna XGBoost (30d Rebal, 30d Fwd - Params...",26173.438619,26.611647,1.074348,1.400950,-43.278393,0.614895,1.069807,16.634597
6,7. Optuna XGBoost (30d Rebal - Dynamic Horizon...,16443.949945,24.155481,0.919933,1.212327,-57.341295,0.421258,1.094907,13.991275
7,8. Optuna XGBoost (15d Rebal - Dynamic Horizon...,13398.895926,23.090460,0.887079,1.186174,-53.770528,0.429426,1.121250,12.729823


## 5. Interactive 8-Curve Multi-Decade Visualizer (Log Scale) & Drawdowns

In [5]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('<b>8-Strategy Walk-Forward Multi-Decade Equity Curves (Log Scale: 2003–2026)</b>',
                                    '<b>Underwater Drawdown Curves (%)</b>'))

palette = [
    '#636EFA',  # 1. SP500
    '#FFA15A',  # 2. Buy & Hold
    '#AB63FA',  # 3. Naive 30_5
    '#19D3F3',  # 4. Naive 30_30
    '#FF6692',  # 5. Naive 15_15
    '#B6E880',  # 6. Optuna 30_30
    '#00CC96',  # 7. Optuna 30_dyn (Hero)
    '#FFDF00'   # 8. Optuna 15_dyn
]

for idx, (name, s) in enumerate(curves.items()):
    c = palette[idx % len(palette)]
    is_hero = ('7.' in name or '6.' in name or '8.' in name)
    fig.add_trace(go.Scatter(
        x=df_master_curves['date'], y=s, name=name,
        line=dict(color=c, width=3.0 if is_hero else 1.8)
    ), row=1, col=1)
    
    dd = ((s - s.cummax()) / s.cummax()) * 100.0
    fig.add_trace(go.Scatter(
        x=df_master_curves['date'], y=dd, name=f"{name} DD", showlegend=False,
        line=dict(color=c, width=1.5)
    ), row=2, col=1)

fig.update_yaxes(type="log", row=1, col=1, title="<b>Portfolio Value ($ Log Scale)</b>")
fig.update_yaxes(row=2, col=1, title="<b>Drawdown (%)</b>")

fig.update_layout(
    template='plotly_dark', width=1200, height=850,
    title='<b>Comprehensive 8-Strategy Benchmark: Passive vs. Naive vs. Purged Nested Optuna Dynamic</b>',
    margin=dict(l=60, r=320, t=80, b=60),
    legend=dict(orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02, title=dict(text='<b>Strategy / Model</b>'))
)
fig.show()

## 6. Dynamic Horizon & Hyperparameter Evolution Over Time

In [6]:
df_p30 = pd.DataFrame(optuna_30_params)
print("=== 30-DAY REBALANCE: DYNAMIC HORIZON & PARAMETER SELECTION DISTRIBUTION ===")
print("Chosen Forward Horizon Days Distribution (Counts):")
print(df_p30['chosen_horizon'].value_counts())
print("\nSample Trajectory (First 5 & Last 5 Cycles):")
print(df_p30[['date', 'chosen_horizon', 'max_depth', 'learning_rate', 'n_estimators', 'subsample']].head(5).to_string(index=False))
print("...")
print(df_p30[['date', 'chosen_horizon', 'max_depth', 'learning_rate', 'n_estimators', 'subsample']].tail(5).to_string(index=False))

=== 30-DAY REBALANCE: DYNAMIC HORIZON & PARAMETER SELECTION DISTRIBUTION ===
Chosen Forward Horizon Days Distribution (Counts):
chosen_horizon
5     68
10    34
15    33
30    22
60    15
20    13
45     7
Name: count, dtype: int64

Sample Trajectory (First 5 & Last 5 Cycles):
      date  chosen_horizon  max_depth  learning_rate  n_estimators  subsample
2003-02-11              15          6       0.030510            46   0.942993
2003-03-27              60          5       0.027722            52   0.788583
2003-05-12               5          5       0.027155            39   0.921989
2003-06-25              15          3       0.035513            37   0.936893
2003-08-08               5          5       0.047464            47   0.982371
...
      date  chosen_horizon  max_depth  learning_rate  n_estimators  subsample
2026-02-26              15          5       0.064652            49   0.988956
2026-04-13              15          4       0.029586            52   0.902144
2026-05-27      

## 7. Export 8-Strategy Walk-Forward Benchmark to Excel

In [7]:
out_path = os.path.join(LOCAL_DATA_DIR, "eight_strategy_purged_nested_optuna_walkforward_simulation_poc.xlsx")
with pd.ExcelWriter(out_path) as writer:
    df_master_curves.to_excel(writer, sheet_name='daily_equity_curves', index=False)
    df_performance_table.to_excel(writer, sheet_name='performance_summary', index=False)
    df_p30.to_excel(writer, sheet_name='optuna_30d_dynamic_params', index=False)
    pd.DataFrame(optuna_15_params).to_excel(writer, sheet_name='optuna_15d_dynamic_params', index=False)

print(f"💾 Successfully exported 8-Strategy Walk-Forward Benchmark to: {out_path}")

💾 Successfully exported 8-Strategy Walk-Forward Benchmark to: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched\eight_strategy_purged_nested_optuna_walkforward_simulation_poc.xlsx
